# Set up

In [1]:
!pip install yfinance

In [2]:
pip install fredapi

Note: you may need to restart the kernel to use updated packages.


# Analysis

In [3]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fredapi import Fred

In [4]:
# Initialize
fred = Fred(api_key='c658af8e4a4241acb757a19536ab8f9d')

In [5]:
def get_expected_market_return(growth_rate=0.06):
    spy = yf.Ticker("SPY")
    div_yield = spy.info.get("dividendYield", 0)
    if div_yield > 1:
        div_yield = div_yield / 100
    return div_yield + growth_rate


In [6]:
def get_cost_of_debt(ticker_symbol):
    ticker = yf.Ticker(ticker_symbol)
    try:
        income_stmt = ticker.financials
        balance_sheet = ticker.balance_sheet
        if "Interest Expense" not in income_stmt.index:
            return 0
        interest_expense_series = income_stmt.loc["Interest Expense"].dropna()
        if interest_expense_series.empty:
            return 0
        interest_expense = abs(interest_expense_series.iloc[0])
        short_term_debt = balance_sheet.loc["Short Term Debt"].iloc[0] if "Short Term Debt" in balance_sheet.index else 0
        long_term_debt = balance_sheet.loc["Long Term Debt"].iloc[0] if "Long Term Debt" in balance_sheet.index else 0
        total_debt = short_term_debt + long_term_debt
        if total_debt == 0:
            return 0
        return interest_expense / total_debt
    except:
        return 0

In [7]:
def run_dcf(ticker_symbol):
    ticker = yf.Ticker(ticker_symbol)
    cash_flow = ticker.cashflow
    balance_sheet = ticker.balance_sheet
    info = ticker.info

    try:
        op_cash_flow = cash_flow.loc["Operating Cash Flow"]
        capex = cash_flow.loc["Capital Expenditure"]
        fcf = (op_cash_flow - capex).sort_index(ascending=True)
        fcf_clean = fcf.astype(float)
        fcf_growth = fcf_clean.pct_change().dropna()
        avg_growth = fcf_growth.mean()
    except:
        return "FCF calculation failed."

    # Risk-free rate from FRED (10-year Treasury)
    yield_data = fred.get_series_latest_release('GS10')
    risk_free_rate = yield_data.iloc[-1] / 100

    expected_market_return = get_expected_market_return()
    beta = info.get("beta", 1.0)
    market_risk_premium = expected_market_return - risk_free_rate
    cost_of_equity = risk_free_rate + beta * market_risk_premium
    cost_of_debt = get_cost_of_debt(ticker_symbol)
    tax_rate = info.get("effectiveTaxRate", 0.21)

    market_cap = info.get("marketCap", 0)
    shares_outstanding = info.get("sharesOutstanding", 0)
    current_price = info.get("currentPrice", 0)
    if market_cap == 0 and shares_outstanding and current_price:
        market_cap = shares_outstanding * current_price

    short_debt = balance_sheet.loc["Short Term Debt"].iloc[0] if "Short Term Debt" in balance_sheet.index else 0
    long_debt = balance_sheet.loc["Long Term Debt"].iloc[0] if "Long Term Debt" in balance_sheet.index else 0
    total_debt = float(short_debt + long_debt)
    total_capital = float(market_cap + total_debt)

    if total_capital == 0:
        return "Capital structure invalid."

    equity_weight = market_cap / total_capital
    debt_weight = total_debt / total_capital
    wacc = (equity_weight * cost_of_equity) + (debt_weight * cost_of_debt * (1 - tax_rate))
    discount_rate = wacc

    fcf_latest = fcf.iloc[-1]
    projection_years = 5
    fcf_projection = [fcf_latest * (1 + avg_growth) ** i for i in range(1, projection_years + 1)]

    terminal_growth = 0.03
    terminal_value = fcf_projection[-1] * (1 + terminal_growth) / (discount_rate - terminal_growth)
    cash_flows = fcf_projection + [terminal_value]
    discounted = [cf / (1 + discount_rate) ** i for i, cf in enumerate(cash_flows, 1)]
    enterprise_value = sum(discounted)

    try:
        cash = balance_sheet.loc["Cash"].iloc[0] if "Cash" in balance_sheet.index else 0
        equity_value = enterprise_value - total_debt + cash
        fair_price = equity_value / shares_outstanding if shares_outstanding else 0
        
        print(f"{ticker_symbol} Fair Value Estimate: $ {fair_price:.2f}")
        #return round(fair_price, 2)
    except:
        return "Equity value calculation failed."


In [8]:
run_dcf("AAPL")
run_dcf("TSLA")

AAPL Fair Value Estimate: $ 182.90
TSLA Fair Value Estimate: $ 127.23


In [13]:
def run_dcf_sensitivity(ticker_symbol, growth_rates, discount_rates):
    ticker = yf.Ticker(ticker_symbol)
    cash_flow = ticker.cashflow
    balance_sheet = ticker.balance_sheet
    info = ticker.info

    try:
        op_cash_flow = cash_flow.loc["Operating Cash Flow"]
        capex = cash_flow.loc["Capital Expenditure"]
        fcf = (op_cash_flow - capex).sort_index(ascending=True)
        fcf_latest = fcf.iloc[-1]
    except:
        return "FCF calculation failed."

    try:
        income_stmt = ticker.financials
        if "Interest Expense" not in income_stmt.index:
            return "Missing Interest Expense"
        interest_expense_series = income_stmt.loc["Interest Expense"].dropna()
        if interest_expense_series.empty:
            return "Empty Interest Expense"
        interest_expense = abs(interest_expense_series.iloc[0])
        short_term_debt = balance_sheet.loc["Short Term Debt"].iloc[0] if "Short Term Debt" in balance_sheet.index else 0
        long_term_debt = balance_sheet.loc["Long Term Debt"].iloc[0] if "Long Term Debt" in balance_sheet.index else 0
        total_debt = short_term_debt + long_term_debt
        cost_of_debt = interest_expense / total_debt if total_debt else 0
    except:
        return "Cost of debt failed"

    try:
        yield_data = fred.get_series_latest_release('GS10')
        risk_free_rate = yield_data.iloc[-1] / 100
    except:
        return "Risk-free rate fetch failed"

    expected_market_return = get_expected_market_return()
    beta = info.get("beta", 1.0)
    market_risk_premium = expected_market_return - risk_free_rate
    cost_of_equity = risk_free_rate + beta * market_risk_premium
    tax_rate = info.get("effectiveTaxRate", 0.21)

    market_cap = info.get("marketCap", 0)
    shares_outstanding = info.get("sharesOutstanding", 0)
    current_price = info.get("currentPrice", 0)
    if market_cap == 0 and shares_outstanding and current_price:
        market_cap = shares_outstanding * current_price

    total_capital = market_cap + total_debt
    equity_weight = market_cap / total_capital
    debt_weight = total_debt / total_capital
    wacc = (equity_weight * cost_of_equity) + (debt_weight * cost_of_debt * (1 - tax_rate))

    terminal_growth = 0.03
    projection_years = 5

    sensitivity_df = pd.DataFrame(index=[f"{int(g*100)}%" for g in growth_rates],
                                   columns=[f"{int(r*100)}%" for r in discount_rates])

    for g in growth_rates:
        for r in discount_rates:
            fcf_projection = [fcf_latest * (1 + g) ** i for i in range(1, projection_years + 1)]
            terminal_value = fcf_projection[-1] * (1 + terminal_growth) / (r - terminal_growth)
            cash_flows = fcf_projection + [terminal_value]
            discounted = [cf / (1 + r) ** i for i, cf in enumerate(cash_flows, 1)]
            enterprise_value = sum(discounted)

            try:
                cash = balance_sheet.loc["Cash"].iloc[0] if "Cash" in balance_sheet.index else 0
                equity_value = enterprise_value - total_debt + cash
                fair_price = equity_value / shares_outstanding if shares_outstanding else 0
                sensitivity_df.loc[f"{int(g*100)}%", f"{int(r*100)}%"] = round(fair_price, 2)
            except:
                sensitivity_df.loc[f"{int(g*100)}%", f"{int(r*100)}%"] = "Err"
    print("DCF Sensitivity Table:")
    print(sensitivity_df)
    return sensitivity_df

In [14]:
growth_rates = np.linspace(0.02, 0.06, 5)  # 2% to 6%
discount_rates = np.linspace(0.06, 0.10, 5)  # 6% to 10%

run_dcf_sensitivity("AAPL", growth_rates, discount_rates)


DCF Sensitivity Table:
        6%      7%      8%      9%     10%
2%  259.39  192.24  152.06  125.35  106.33
3%  284.78  210.81  166.56  137.16  116.23
3%  284.78  210.81  166.56  137.16  116.23
5%  298.21  220.63  174.23  143.39  121.45
6%  312.14  230.81  182.17  149.86  126.87


,6%,7%,8%,9%,10%
2%,259.39,192.24,152.06,125.35,106.33
3%,284.78,210.81,166.56,137.16,116.23
3%,284.78,210.81,166.56,137.16,116.23
5%,298.21,220.63,174.23,143.39,121.45
6%,312.14,230.81,182.17,149.86,126.87
